In [2]:
import numpy as np
import pandas as pd

BUDGET = 1_000_000


def optimize_manual_portfolio(expected_returns, max_total_volume_pct=100):
    """
    expected_returns:
        dict of product -> expected return

        Positive return means long.
        Negative return means short.

        Example:
        {
            "SULFUR_LTD": 0.35,
            "LAVA_CAKES": -0.30,
        }
    """

    rows = []

    for product, r in expected_returns.items():
        if r == 0 or pd.isna(r):
            continue

        rows.append(
            {
                "product": product,
                "side": "BUY" if r > 0 else "SELL",
                "expected_return": r,
                "edge": abs(r),
            }
        )

    df = pd.DataFrame(rows)

    if df.empty:
        return df, {}

    df = df.sort_values("edge", ascending=False).reset_index(drop=True)

    edges = df["edge"].values
    max_total_w = max_total_volume_pct / 100

    # Unconstrained optimum: w_i = edge_i / 2
    w_unconstrained = edges / 2

    if w_unconstrained.sum() <= max_total_w:
        w = w_unconstrained
    else:
        # Budget-constrained case:
        # w_i = max(0, (edge_i - lambda) / 2)
        lo, hi = 0, edges.max()

        for _ in range(100):
            lam = (lo + hi) / 2
            w_candidate = np.maximum(0, (edges - lam) / 2)

            if w_candidate.sum() > max_total_w:
                lo = lam
            else:
                hi = lam

        w = np.maximum(0, (edges - hi) / 2)

    df["volume_pct"] = 100 * w
    df["fee"] = BUDGET * w**2
    df["gross_expected_pnl"] = BUDGET * w * df["edge"]
    df["net_expected_pnl"] = df["gross_expected_pnl"] - df["fee"]

    df = df[df["volume_pct"] > 1e-9].copy()

    summary = {
        "total_volume_pct": df["volume_pct"].sum(),
        "total_fee": df["fee"].sum(),
        "total_gross_expected_pnl": df["gross_expected_pnl"].sum(),
        "total_net_expected_pnl": df["net_expected_pnl"].sum(),
    }

    return df, summary


expected_returns = {
    "SULFUR_LTD": 0.35,
    "LAVA_CAKES": -0.30,
    "THERMALITE_CORES": 0.25,
    "PYROFLEX_CELL": -0.20,
    "MAGMA_INK": 0.18,
    "SCORIA_PASTE": 0.15,
    "ASHES_OF_THE_PHOENIX": -0.12,
    "VOLCANIC_INCENSE": 0.10,
    "OBSIDIAN_CUTLERY": 0.08,
}

portfolio, summary = optimize_manual_portfolio(expected_returns)

display(portfolio)
print(summary)

,product,side,expected_return,edge,volume_pct,fee,gross_expected_pnl,net_expected_pnl
0,SULFUR_LTD,BUY,0.35,0.35,17.5,30625.0,61250.0,30625.0
1,LAVA_CAKES,SELL,-0.30,0.30,15.0,22500.0,45000.0,22500.0
2,THERMALITE_CORES,BUY,0.25,0.25,12.5,15625.0,31250.0,15625.0
3,PYROFLEX_CELL,SELL,-0.20,0.20,10.0,10000.0,20000.0,10000.0
4,MAGMA_INK,BUY,0.18,0.18,9.0,8100.0,16200.0,8100.0
5,SCORIA_PASTE,BUY,0.15,0.15,7.5,5625.0,11250.0,5625.0
6,ASHES_OF_THE_PHOENIX,SELL,-0.12,0.12,6.0,3600.0,7200.0,3600.0
7,VOLCANIC_INCENSE,BUY,0.10,0.10,5.0,2500.0,5000.0,2500.0
8,OBSIDIAN_CUTLERY,BUY,0.08,0.08,4.0,1600.0,3200.0,1600.0


{'total_volume_pct': np.float64(86.5), 'total_fee': np.float64(100175.0), 'total_gross_expected_pnl': np.float64(200350.0), 'total_net_expected_pnl': np.float64(100175.0)}


In [3]:
import numpy as np
import pandas as pd

P = "SNACKPACK_PISTACHIO"
R = "SNACKPACK_RASPBERRY"


def prepare_pair_df(prices):
    df = prices.copy()
    df["product"] = df["product"].astype(str).str.strip()

    if "mid_price" not in df.columns:
        bid_cols = [c for c in df.columns if c.startswith("bid_price_")]
        ask_cols = [c for c in df.columns if c.startswith("ask_price_")]
        df["best_bid"] = df[bid_cols].replace(0, np.nan).max(axis=1)
        df["best_ask"] = df[ask_cols].replace(0, np.nan).min(axis=1)
        df["mid_price"] = (df["best_bid"] + df["best_ask"]) / 2

    df["mid_price"] = pd.to_numeric(df["mid_price"], errors="coerce")
    df.loc[df["mid_price"] == 0, "mid_price"] = np.nan

    df = df[df["product"].isin([P, R])].copy()
    df = df.sort_values(["day", "timestamp", "product"])

    min_day = df["day"].min()
    day_span = df.groupby("day")["timestamp"].max().max() + 1
    df["global_time"] = (df["day"] - min_day) * day_span + df["timestamp"]

    df["mid_price"] = df.groupby("product")["mid_price"].transform(
        lambda s: s.ffill().bfill()
    )

    pair = df.pivot_table(
        index="global_time",
        columns="product",
        values="mid_price",
        aggfunc="last",
    ).dropna()

    return pair


def evaluate_pair_signal(
    pair, spread_type, mode, window=120, entry_z=1.5, horizon=10
):
    log_p = np.log(pair[P])
    log_r = np.log(pair[R])

    if spread_type == "SUM":
        spread = log_p + log_r
    elif spread_type == "DIFF":
        spread = log_p - log_r
    else:
        raise ValueError("spread_type must be SUM or DIFF")

    roll_mean = spread.rolling(window).mean().shift(1)
    roll_std = spread.rolling(window).std().shift(1)

    z = (spread - roll_mean) / (roll_std + 1e-12)

    raw_signal = pd.Series(0, index=spread.index)
    raw_signal[z > entry_z] = 1
    raw_signal[z < -entry_z] = -1

    if mode == "REVERSION":
        spread_position = -raw_signal
    elif mode == "MOMENTUM":
        spread_position = raw_signal
    else:
        raise ValueError("mode must be REVERSION or MOMENTUM")

    future_spread_change = spread.shift(-horizon) - spread

    # Positive means the signal predicted the spread direction correctly.
    edge = spread_position * future_spread_change
    edge = edge[spread_position != 0].dropna()

    if len(edge) == 0:
        return None

    return {
        "spread_type": spread_type,
        "mode": mode,
        "window": window,
        "entry_z": entry_z,
        "horizon": horizon,
        "num_trades": len(edge),
        "avg_edge": edge.mean(),
        "median_edge": edge.median(),
        "win_rate": (edge > 0).mean(),
        "total_edge": edge.sum(),
    }


pair = prepare_pair_df(prices)

records = []

for spread_type in ["SUM", "DIFF"]:
    for mode in ["REVERSION", "MOMENTUM"]:
        for window in [60, 120, 200, 300]:
            for entry_z in [1.0, 1.25, 1.5, 1.75, 2.0]:
                for horizon in [5, 10, 20, 50]:
                    out = evaluate_pair_signal(
                        pair,
                        spread_type=spread_type,
                        mode=mode,
                        window=window,
                        entry_z=entry_z,
                        horizon=horizon,
                    )
                    if out is not None:
                        records.append(out)

signal_test = pd.DataFrame(records)

display(
    signal_test.sort_values(
        ["avg_edge", "win_rate", "total_edge"], ascending=False
    ).head(30)
)

,spread_type,mode,window,entry_z,horizon,num_trades,avg_edge,median_edge,win_rate,total_edge
239,DIFF,REVERSION,300,2.00,50,4265,0.001374,0.001131,0.552638,5.859970
219,DIFF,REVERSION,200,2.00,50,4397,0.000988,0.000804,0.540141,4.345039
235,DIFF,REVERSION,300,1.75,50,6631,0.000742,0.000466,0.520133,4.922016
215,DIFF,REVERSION,200,1.75,50,6741,0.000739,0.000563,0.528408,4.978834
238,DIFF,REVERSION,300,2.00,20,4265,0.000696,0.000711,0.554045,2.970516
199,DIFF,REVERSION,120,2.00,50,4530,0.000608,0.000427,0.520309,2.754555
195,DIFF,REVERSION,120,1.75,50,6931,0.000575,0.000372,0.517241,3.982098
231,DIFF,REVERSION,300,1.50,50,9611,0.000501,0.000258,0.510561,4.816012
223,DIFF,REVERSION,300,1.00,50,16536,0.000490,0.000183,0.508043,8.095557
227,DIFF,REVERSION,300,1.25,50,12980,0.000487,0.000223,0.509707,6.318413


In [4]:
import numpy as np
import pandas as pd
import itertools

SNACKS = [
    "SNACKPACK_CHOCOLATE",
    "SNACKPACK_VANILLA",
    "SNACKPACK_PISTACHIO",
    "SNACKPACK_STRAWBERRY",
    "SNACKPACK_RASPBERRY",
]


def prepare_mid_matrix(prices):
    df = prices.copy()
    df["product"] = df["product"].astype(str).str.strip()

    if "mid_price" not in df.columns:
        bid_cols = [c for c in df.columns if c.startswith("bid_price_")]
        ask_cols = [c for c in df.columns if c.startswith("ask_price_")]
        df["best_bid"] = df[bid_cols].replace(0, np.nan).max(axis=1)
        df["best_ask"] = df[ask_cols].replace(0, np.nan).min(axis=1)
        df["mid_price"] = (df["best_bid"] + df["best_ask"]) / 2

    df["mid_price"] = pd.to_numeric(df["mid_price"], errors="coerce")
    df.loc[df["mid_price"] == 0, "mid_price"] = np.nan

    df = df[df["product"].isin(SNACKS)].copy()
    df = df.sort_values(["day", "timestamp", "product"])

    min_day = df["day"].min()
    day_span = df.groupby("day")["timestamp"].max().max() + 1
    df["global_time"] = (df["day"] - min_day) * day_span + df["timestamp"]

    df["mid_price"] = df.groupby("product")["mid_price"].transform(
        lambda s: s.ffill().bfill()
    )

    mids = df.pivot_table(
        index=["day", "global_time"],
        columns="product",
        values="mid_price",
        aggfunc="last",
    ).dropna()

    return mids


def evaluate_static_basket(mids, targets, name="basket"):
    rows = []

    for day, g in mids.groupby(level=0):
        g = g.droplevel(0)

        pnl_series = pd.Series(0.0, index=g.index)

        for product, pos in targets.items():
            pnl_series += pos * (g[product] - g[product].iloc[0])

        rows.append(
            {
                "name": name,
                "day": day,
                "final_pnl": pnl_series.iloc[-1],
                "max_pnl": pnl_series.max(),
                "min_pnl": pnl_series.min(),
                "drawdown_from_peak": pnl_series.max() - pnl_series.iloc[-1],
                "pnl_vol": pnl_series.diff().std(),
            }
        )

    return pd.DataFrame(rows)


mids = prepare_mid_matrix(prices)

original_targets = {
    "SNACKPACK_CHOCOLATE": -10,
    "SNACKPACK_VANILLA": 10,
    "SNACKPACK_PISTACHIO": -10,
    "SNACKPACK_STRAWBERRY": 10,
    "SNACKPACK_RASPBERRY": 10,
}

core_targets = {
    "SNACKPACK_CHOCOLATE": -10,
    "SNACKPACK_VANILLA": 10,
    "SNACKPACK_PISTACHIO": -10,
    "SNACKPACK_STRAWBERRY": 0,
    "SNACKPACK_RASPBERRY": 10,
}

balanced_targets = {
    "SNACKPACK_CHOCOLATE": -10,
    "SNACKPACK_VANILLA": 10,
    "SNACKPACK_PISTACHIO": -10,
    "SNACKPACK_STRAWBERRY": 5,
    "SNACKPACK_RASPBERRY": 5,
}

results = pd.concat(
    [
        evaluate_static_basket(mids, original_targets, "original"),
        evaluate_static_basket(mids, core_targets, "core_no_strawberry"),
        evaluate_static_basket(mids, balanced_targets, "balanced_straw_rasp"),
    ]
)

display(results)
display(
    results.groupby("name")
    .mean(numeric_only=True)
    .sort_values("final_pnl", ascending=False)
)

,name,day,final_pnl,max_pnl,min_pnl,drawdown_from_peak,pnl_vol
0,original,2,13015.0,18060.0,-1285.0,5045.0,140.041318
1,original,3,4180.0,11590.0,-5675.0,7410.0,138.906477
2,original,4,10445.0,13910.0,-4865.0,3465.0,137.789309
0,core_no_strawberry,2,8655.0,13640.0,-3385.0,4985.0,182.794861
1,core_no_strawberry,3,605.0,8735.0,-12225.0,8130.0,180.279391
2,core_no_strawberry,4,9470.0,13665.0,-6990.0,4195.0,177.503759
0,balanced_straw_rasp,2,9637.5,15687.5,-1415.0,6050.0,139.035946
1,balanced_straw_rasp,3,2942.5,10207.5,-6675.0,7265.0,137.499878
2,balanced_straw_rasp,4,9057.5,12820.0,-4695.0,3762.5,136.166684


,day,final_pnl,max_pnl,min_pnl,drawdown_from_peak,pnl_vol
name,,,,,,
original,3.0,9213.333333,14520.000000,-3941.666667,5306.666667,138.912368
balanced_straw_rasp,3.0,7212.500000,12905.000000,-4261.666667,5692.500000,137.567503
core_no_strawberry,3.0,6243.333333,12013.333333,-7533.333333,5770.000000,180.192671


In [6]:
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go

# ============================================================
# 1. Load / prepare Round 5 prices
# ============================================================


def load_prices_if_needed():
    """
    Uses existing `prices` if already defined.
    Otherwise tries to load prices_round_5_day_*.csv from common folders.
    """
    global prices

    try:
        prices
        print("Using existing `prices` DataFrame.")
        return prices.copy()
    except NameError:
        pass

    possible_dirs = [
        Path("data"),
        Path("round_5/data"),
        Path("."),
    ]

    files = []
    for d in possible_dirs:
        files.extend(sorted(d.glob("prices_round_5_day_*.csv")))

    files = sorted(set(files))

    if not files:
        raise FileNotFoundError(
            "Could not find prices_round_5_day_*.csv. Check your working directory."
        )

    dfs = []
    for f in files:
        df = pd.read_csv(f, sep=";")
        df.columns = [c.strip() for c in df.columns]
        dfs.append(df)

    out = pd.concat(dfs, ignore_index=True)
    print(f"Loaded {len(files)} files, {len(out):,} rows.")
    return out


def prepare_mid_matrix(prices_df):
    df = prices_df.copy()
    df.columns = [c.strip() for c in df.columns]

    if "symbol" in df.columns and "product" not in df.columns:
        df = df.rename(columns={"symbol": "product"})

    df["product"] = df["product"].astype(str).str.strip()
    df["day"] = df["day"].astype(int)
    df["timestamp"] = df["timestamp"].astype(int)

    # Reconstruct mid if needed
    if "mid_price" not in df.columns:
        bid_cols = [c for c in df.columns if c.startswith("bid_price_")]
        ask_cols = [c for c in df.columns if c.startswith("ask_price_")]

        df["best_bid"] = df[bid_cols].replace(0, np.nan).max(axis=1)
        df["best_ask"] = df[ask_cols].replace(0, np.nan).min(axis=1)
        df["mid_price"] = (df["best_bid"] + df["best_ask"]) / 2

    df["mid_price"] = pd.to_numeric(df["mid_price"], errors="coerce")
    df.loc[df["mid_price"] == 0, "mid_price"] = np.nan

    df = df.sort_values(["day", "timestamp", "product"])

    # Fill missing mid prices inside each product/day
    df["mid_price"] = df.groupby(["day", "product"])["mid_price"].transform(
        lambda s: s.ffill().bfill()
    )

    mids = df.pivot_table(
        index=["day", "timestamp"],
        columns="product",
        values="mid_price",
        aggfunc="last",
    )

    return mids


prices_df = load_prices_if_needed()
mids = prepare_mid_matrix(prices_df)

print("Mid matrix shape:", mids.shape)
display(mids.head())


# ============================================================
# 2. Define baskets to test
# ============================================================

BASKETS = {
    "pebbles_full": {
        "PEBBLES_XS": -10,
        "PEBBLES_S": -10,
        "PEBBLES_M": 10,
        "PEBBLES_L": 10,
        "PEBBLES_XL": 10,
    },
    "microchips_full": {
        "MICROCHIP_CIRCLE": 10,
        "MICROCHIP_OVAL": -10,
        "MICROCHIP_SQUARE": 10,
        "MICROCHIP_RECTANGLE": -10,
        "MICROCHIP_TRIANGLE": -10,
    },
    "uv_full": {
        "UV_VISOR_AMBER": -10,
        "UV_VISOR_ORANGE": -10,
        "UV_VISOR_RED": 10,
        "UV_VISOR_MAGENTA": 10,
    },
    "refined_3_basket": {
        "PEBBLES_XS": -10,
        "PEBBLES_S": -10,
        "PEBBLES_XL": 10,
        "MICROCHIP_OVAL": -10,
        "MICROCHIP_RECTANGLE": -10,
        "UV_VISOR_AMBER": -10,
        "UV_VISOR_RED": 10,
    },
    "snack_original": {
        "SNACKPACK_CHOCOLATE": -10,
        "SNACKPACK_VANILLA": 10,
        "SNACKPACK_PISTACHIO": -10,
        "SNACKPACK_STRAWBERRY": 10,
        "SNACKPACK_RASPBERRY": 10,
    },
}


# ============================================================
# 3. Basket value and PnL functions
# ============================================================


def basket_value_series(mids, targets):
    """
    Basket value = sum(position_i * mid_i).
    If this rises, this basket makes mark-to-market PnL.
    """
    products = list(targets.keys())
    missing = [p for p in products if p not in mids.columns]

    if missing:
        raise ValueError(f"Missing products in mids: {missing}")

    value = pd.Series(0.0, index=mids.index)

    for product, pos in targets.items():
        value += pos * mids[product]

    return value.dropna()


def basket_pnl_by_day(mids, targets):
    value = basket_value_series(mids, targets)

    rows = []
    pnl_series_all = []

    for day, g in value.groupby(level=0):
        g = g.droplevel(0)
        pnl = g - g.iloc[0]

        rows.append(
            {
                "day": day,
                "final_pnl": pnl.iloc[-1],
                "max_pnl": pnl.max(),
                "min_pnl": pnl.min(),
                "drawdown_from_peak": pnl.max() - pnl.iloc[-1],
                "pnl_vol": pnl.diff().std(),
            }
        )

        temp = pnl.copy()
        temp.index = pd.MultiIndex.from_product(
            [[day], temp.index], names=["day", "timestamp"]
        )
        pnl_series_all.append(temp)

    return pd.DataFrame(rows), pd.concat(pnl_series_all)


# ============================================================
# 4. Test whether a signal predicts future basket returns
# ============================================================


def make_momentum_signal(value, lookback=100, vol_lookback=100):
    """
    Signal = recent basket momentum normalized by recent volatility.

    Positive signal means basket has recently moved in the direction of the target positions.
    Negative signal means basket has recently moved against the target positions.
    """
    signal_parts = []

    for day, g in value.groupby(level=0):
        x = g.droplevel(0)

        mom = x - x.shift(lookback)
        vol = x.diff().rolling(vol_lookback).std()

        sig = mom / (vol * np.sqrt(lookback) + 1e-9)
        sig.index = pd.MultiIndex.from_product(
            [[day], sig.index], names=["day", "timestamp"]
        )

        signal_parts.append(sig)

    return pd.concat(signal_parts)


def test_signal_predictiveness(
    value, signal, horizons=(10, 25, 50, 100), bins=5
):
    """
    For each signal bin, check future basket return.

    A useful momentum signal should look like:
        high signal bin -> higher future_return
        low signal bin  -> lower future_return
    """
    rows = []

    for h in horizons:
        pieces = []

        for day, g in value.groupby(level=0):
            x = g.droplevel(0)
            s = signal.xs(day, level=0)

            future = x.shift(-h) - x

            temp = pd.DataFrame(
                {
                    "signal": s,
                    "future_return": future,
                }
            ).dropna()

            temp["day"] = day
            pieces.append(temp)

        df = pd.concat(pieces, ignore_index=True)

        try:
            df["signal_bin"] = pd.qcut(
                df["signal"], q=bins, labels=False, duplicates="drop"
            )
        except ValueError:
            continue

        grouped = (
            df.groupby("signal_bin")
            .agg(
                mean_signal=("signal", "mean"),
                mean_future_return=("future_return", "mean"),
                median_future_return=("future_return", "median"),
                win_rate=("future_return", lambda x: (x > 0).mean()),
                count=("future_return", "size"),
            )
            .reset_index()
        )

        grouped["horizon"] = h
        rows.append(grouped)

    if not rows:
        return pd.DataFrame()

    return pd.concat(rows, ignore_index=True)


def summarize_predictiveness(test_df):
    """
    Checks whether higher signal bins tend to have better future returns.
    """
    rows = []

    for h, g in test_df.groupby("horizon"):
        g = g.sort_values("signal_bin")

        if len(g) < 2:
            continue

        low = g.iloc[0]["mean_future_return"]
        high = g.iloc[-1]["mean_future_return"]

        rows.append(
            {
                "horizon": h,
                "low_bin_future_return": low,
                "high_bin_future_return": high,
                "high_minus_low": high - low,
                "best_bin": int(
                    g.loc[g["mean_future_return"].idxmax(), "signal_bin"]
                ),
                "worst_bin": int(
                    g.loc[g["mean_future_return"].idxmin(), "signal_bin"]
                ),
            }
        )

    return pd.DataFrame(rows)


# ============================================================
# 5. Run test for all baskets
# ============================================================

all_perf = []
all_predictiveness = []
all_bin_tables = {}

for name, targets in BASKETS.items():
    print("\n" + "=" * 80)
    print("Basket:", name)
    print("=" * 80)

    value = basket_value_series(mids, targets)
    perf, pnl_series = basket_pnl_by_day(mids, targets)

    display(perf)

    signal = make_momentum_signal(value, lookback=100, vol_lookback=100)
    test_df = test_signal_predictiveness(
        value, signal, horizons=(10, 25, 50, 100), bins=5
    )
    summary = summarize_predictiveness(test_df)

    all_bin_tables[name] = test_df

    print("Signal predictiveness summary:")
    display(summary)

    perf["basket"] = name
    all_perf.append(perf)

    summary["basket"] = name
    all_predictiveness.append(summary)


perf_all = pd.concat(all_perf, ignore_index=True)
pred_all = pd.concat(all_predictiveness, ignore_index=True)

print("\n\n================ Final basket performance summary ================")
display(
    perf_all.groupby("basket")
    .agg(
        avg_final_pnl=("final_pnl", "mean"),
        min_final_pnl=("final_pnl", "min"),
        avg_max_pnl=("max_pnl", "mean"),
        avg_min_pnl=("min_pnl", "mean"),
        avg_drawdown=("drawdown_from_peak", "mean"),
        avg_vol=("pnl_vol", "mean"),
    )
    .sort_values("avg_final_pnl", ascending=False)
)

print(
    "\n\n================ Predictiveness summary across baskets ================"
)
display(pred_all.sort_values(["basket", "horizon"]))

Using existing `prices` DataFrame.
Mid matrix shape: (30000, 50)


product        GALAXY_SOUNDS_BLACK_HOLES  GALAXY_SOUNDS_DARK_MATTER  \
day timestamp                                                         
2   0                            10000.0                    10000.0   
    100                          10007.5                     9988.5   
    200                          10002.5                    10003.5   
    300                          10000.5                     9994.5   
    400                          10005.5                     9986.5   

product        GALAXY_SOUNDS_PLANETARY_RINGS  GALAXY_SOUNDS_SOLAR_FLAMES  \
day timestamp                                                              
2   0                                10000.0                     10000.0   
    100                               9988.5                     10002.5   
    200                               9995.5                     10003.5   
    300                               9997.5                     10012.5   
    400                              10008.5                     10012.5   

product        GALAXY_SOUNDS_SOLAR_WINDS  MICROCHIP_CIRCLE  MICROCHIP_OVAL  \
day timestamp                                                                
2   0                            10000.0           10000.0         10000.0   
    100                          10000.5            9991.5          9999.5   
    200                          10007.5            9993.5          9998.5   
    300                          10021.5            9969.5          9997.5   
    400                          10013.5            9977.5          9996.5   

product        MICROCHIP_RECTANGLE  MICROCHIP_SQUARE  MICROCHIP_TRIANGLE  \
day timestamp                                                              
2   0                      10000.0           10000.0             10000.0   
    100                     9999.5            9999.5             10000.5   
    200                    10001.5            9999.5             10001.5   
    300                    10002.5            9998.5             10001.5   
    400                    10001.5           10000.5             10000.5   

product        OXYGEN_SHAKE_CHOCOLATE  OXYGEN_SHAKE_EVENING_BREATH  \
day timestamp                                                        
2   0                         10000.0                      10000.0   
    100                       10000.0                      10000.0   
    200                       10000.0                      10000.0   
    300                       10000.0                      10000.0   
    400                       10000.0                      10000.0   

product        OXYGEN_SHAKE_GARLIC  OXYGEN_SHAKE_MINT  \
day timestamp                                           
2   0                      10000.0            10000.0   
    100                    10005.5            10008.5   
    200                    10006.5             9998.5   
    300                    10024.5            10005.5   
    400                    10012.5             9997.5   

product        OXYGEN_SHAKE_MORNING_BREATH  PANEL_1X2  PANEL_1X4  PANEL_2X2  \
day timestamp                                                                 
2   0                              10000.0    10000.0    10000.0    10000.0   
    100                             9991.5    10002.5     9996.5     9976.0   
    200                            10003.5    10016.5     9992.5     9991.5   
    300                            10000.5    10010.5    10000.5     9980.0   
    400                             9997.5    10021.5     9991.5     9982.5   

product        PANEL_2X4  PANEL_4X4  PEBBLES_L  PEBBLES_M  PEBBLES_S  \
day timestamp                                                          
2   0            10000.0    10000.0    10000.0    10000.0    10000.0   
    100           9998.5     9997.5     9997.5    10010.5    10010.5   
    200          10004.5    10008.5     9993.5    10040.5    10012.5   
    300           9991.5    10016.5     9981.5    10034.5     9999.5   
    400     


Basket: pebbles_full


,day,final_pnl,max_pnl,min_pnl,drawdown_from_peak,pnl_vol
0,2,55835.0,63650.0,-23645.0,7815.0,423.690455
1,3,27615.0,49950.0,-7645.0,22335.0,422.852068
2,4,35220.0,75590.0,-1610.0,40370.0,422.960516


Signal predictiveness summary:


...

,horizon,low_bin_future_return,high_bin_future_return,high_minus_low,best_bin,worst_bin
0,10,21.749242,11.674250,-10.074992,0,2
1,25,53.778059,31.484388,-22.293671,0,2
2,50,62.302876,49.544839,-12.758037,0,1
3,100,113.938776,52.863946,-61.074830,2,4




================ Final basket performance summary ================


,avg_final_pnl,min_final_pnl,avg_max_pnl,avg_min_pnl,avg_drawdown,avg_vol
basket,,,,,,
refined_3_basket,74433.333333,45155.0,99433.333333,-5255.000000,25000.000000,529.020148
pebbles_full,39556.666667,27615.0,63063.333333,-10966.666667,23506.666667,423.167680
microchips_full,39180.000000,16100.0,48996.666667,-9451.666667,9816.666667,322.906487
uv_full,22565.000000,14310.0,25125.000000,-3545.000000,2560.000000,203.824483
snack_original,9213.333333,4180.0,14520.000000,-3941.666667,5306.666667,138.912368




================ Predictiveness summary across baskets ================


,horizon,low_bin_future_return,high_bin_future_return,high_minus_low,best_bin,worst_bin,basket
4,10,70.881362,66.916077,-3.965285,0,2,microchips_full
5,25,135.305485,150.562869,15.257384,4,3,microchips_full
6,50,253.768190,198.571066,-55.197124,0,3,microchips_full
7,100,481.311224,328.441327,-152.869898,1,3,microchips_full
0,10,88.591170,83.297944,-5.293225,1,3,pebbles_full
1,25,248.143460,159.262447,-88.881013,0,3,pebbles_full
2,50,417.017766,238.421320,-178.596447,0,3,pebbles_full
3,100,776.394558,655.494048,-120.900510,0,3,pebbles_full
12,10,181.552073,75.385912,-106.166161,0,3,refined_3_basket
13,25,479.643038,242.082700,-237.560338,0,3,refined_3_basket
